# Phishing detection model training


Here the necessary steps are copied from the exploratory notebook so model training can be repeated in a concise way without the remarks I made during the exploratory phase.

In [ ]:
import os
import urllib
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, 
    RocCurveDisplay,
    classification_report, 
    confusion_matrix, 
    confusion_matrix_at_thresholds,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

In [ ]:
FOLDER = Path("data")
PATH = FOLDER / "train.csv"

In [ ]:
df = pd.read_csv(PATH)

In [ ]:
def remove_urls_with_conflicting_labels(df: pd.DataFrame) -> pd.DataFrame:
    label_counts = df.groupby("url")["label"].transform("nunique")
    return df[label_counts == 1]


def transform_url(url: str) -> str:
    url = url.rstrip("/ ")
    try:
        parsed = urllib.parse.urlsplit(url)
        url = parsed.netloc + parsed.path
        if parsed.query:
            url = url + "?" + parsed.query

    except ValueError:
        pass
    
    url = urllib.parse.unquote(url)
    return url.lower()

In [ ]:
df = remove_urls_with_conflicting_labels(df)

In [ ]:
df.drop_duplicates("url", inplace=True, ignore_index=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[["url"]], 
    df["label"], 
    test_size=0.2,
    random_state=2112,
    shuffle=True,
    stratify=df["label"],
)

In [ ]:
def transform_urls(urls):
    return [transform_url(url) for url in urls]


model = Pipeline([
    (
        "tfidf", 
        TfidfVectorizer(
            analyzer="char", 
            ngram_range=(2, 4), 
            min_df=3,
            max_df=30_000,
            lowercase=False, 
            sublinear_tf=True,
            use_idf=True,
        )),
    ("classifier", LogisticRegression(C=0.5, max_iter=1000, random_state=2112)),
])

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2112)

y_train_oof_probs = cross_val_predict(
    model, 
    transform_urls(X_train["url"]), 
    y_train, 
    cv=cv, 
    method="predict_proba"
)[:, 1]

## Select threshold

In [ ]:
observed_probability = y_train.mean()  # approx 0.5
target_probability = 1 / 1001  # this is the prevalence of the positive class in real life

pos_weight = target_probability / observed_probability
neg_weight = (1 - target_probability) / (1 - observed_probability)
sample_weight = np.where(y_train == 1, pos_weight, neg_weight)

precision, recall, thresholds = precision_recall_curve(y_train, y_train_oof_probs, sample_weight=sample_weight)

f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-14)
best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]

print(f"Best threshold: {best_threshold:.5f}")
print(f"CV F1: {f1[best_idx]:.5f}")
print(f"Precision: {precision[best_idx]:.5f}")
print(f"Recall: {recall[best_idx]:.5f}")

## Fit final model

In [ ]:
model.fit(transform_urls(X_train["url"]), y_train)

In [ ]:
y_test_prob = model.predict_proba(transform_urls(X_test["url"]))[:, 1]
y_test_pred = (y_test_prob >= best_threshold).astype(np.int32)

## Evaluate the model

In [ ]:
metrics = classification_report(y_test, y_test_pred, digits=3)
cm = confusion_matrix(y_test, y_test_pred)

In [ ]:
print(metrics)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])

disp.plot(cmap=plt.cm.Blues)
plt.show()

In [ ]:
auc = roc_auc_score(y_test, y_test_prob)
print(f"AUC: {auc}")

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_test_prob)

plt.show()

In [ ]:
print("F1 @ 0.5:", f1_score(y_test, y_test_prob >= 0.5))

print(f"F1 @ {best_threshold:.5f}:", f1_score(y_test, y_test_prob >= best_threshold))

## Model export

In [ ]:
os.makedirs("./model", exist_ok=True)
artifact = {
    "model": model,
    "threshold": best_threshold,
}

joblib.dump(artifact, "model/url_phishing_classifier.joblib")